# M4: conditional RBF-kernel copula (diagnostic)

M4 is an exploratory diagnostic, not a primary confirmatory competitor. It embeds each entity feature vector and forms a smooth kernel correlation:

$$h_k=g_\theta(v_k),\qquad K_{ab}=\exp\left(-\frac{\lVert h_a-h_b\rVert_2^2}{2\ell^2}\right)+\delta_{ab}\nu.$$

After normalization and stabilization, $K$ becomes $R$. The RBF construction is positive semidefinite and permutation equivariant, but it restricts off-diagonal dependence to be non-negative. That restriction makes it useful for diagnosis, not a general alternative to M1--M3.

In [ ]:
from pathlib import Path
import sys, numpy as np
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
from simcast.config import SimcastConfig, deep_merge, load_config
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

CONFIG_FILE = 'configs/powertech2027/transformer.yaml'
OVERRIDES = ()
TRAIN_IF_MISSING = False
RUN_EVALUATION = False
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm4_kernel_diagnostic'
base = load_config(PROJECT_ROOT / CONFIG_FILE, overrides=OVERRIDES)
config = SimcastConfig.model_validate(deep_merge(base.model_dump(mode='python'), {'dependence': {'method': 'conditional_kernel'}}))
CACHE_DIR = PROJECT_ROOT / config.output.cache_dir / (config.output.cache_name or f'liander2024_{config.data.entity_type}')
library = load_pit_library(CACHE_DIR, access='training'); ds = library.dataset
entity_ids = [str(x) for x in ds.entity_id.values]; K_g = len(entity_ids)
assert entity_ids == config.protocol.ordered_entity_ids and K_g == config.protocol.entity_count
print(f'group={config.data.entity_type}, K_g={K_g}, diagnostic={config.dependence.conditional_kernel.smoke_only}')

## What can and cannot be inferred

Close embedded feature vectors imply stronger positive residual-rank association. The construction cannot express negative pairwise correlations before normalization, asymmetric tail dependence, or a temporal copula. A favourable M4 score would therefore be evidence about this restricted smooth kernel family only; an unfavourable score would not refute conditional dependence more generally.

The full-group requirement remains unchanged. Kernel entries are computed for every pair in $\mathcal E_g\times\mathcal E_g$, and an invalid member invalidates the complete case.

In [ ]:
z = np.asarray(ds['pit_z'].values)
complete = np.isfinite(z).all(axis=1)
print('complete vectors available for the diagnostic:', int(complete.sum()))
print('kernel embedding dimension:', config.dependence.conditional_kernel.embedding_dim)
print('initial length scale:', config.dependence.conditional_kernel.initial_length_scale)
print('nugget:', config.dependence.conditional_kernel.nugget)
assert z.shape[1] == K_g

## Optional bounded run

The supplied M4 configuration is deliberately bounded by its smoke settings unless they are explicitly changed in the YAML. The cell below follows the identical CLI training/evaluation route but remains disabled. It should not be used to replace the predeclared M0--M3 comparison or to select a paper method from test performance.

In [ ]:
run_dir = OUTPUT_DIR / 'conditional_kernel'
if TRAIN_IF_MISSING and not run_dir.exists():
    run_dir = train_from_config(config, cache_dir=CACHE_DIR, output_dir=run_dir)
if RUN_EVALUATION:
    if not run_dir.exists(): raise FileNotFoundError('Set TRAIN_IF_MISSING=True or choose an existing M4 run.')
    evaluate_from_config(config, methods=('conditional_kernel',), method_runs={'conditional_kernel': run_dir}, cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR / 'evaluation')
else:
    print('Read-only mode: no M4 training or evaluation artifact is written.')

In [ ]:
import pandas as pd
from IPython.display import display
metrics_file = OUTPUT_DIR / 'evaluation' / 'metrics_by_lead.csv'
if metrics_file.is_file():
    metrics = pd.read_csv(metrics_file)
    display(metrics.groupby('method', as_index=False).mean(numeric_only=True))
    metrics.pivot(index='lead', columns='method', values='mean_pinball').plot(title='M4 aggregate pinball loss by lead')
else:
    print('No notebook evaluation table yet. Read-only inspection does not create one.')